## About Dataset

### Context

This dataset contains a subset of book reviews from the **Amazon Kindle Store** category.

### Content

The dataset is a **5-core dataset** of product reviews from the Amazon Kindle Store category collected from **May 1996 to July 2014**.

It contains **982,619 review entries**. Each reviewer has at least 5 reviews, and each product has at least 5 reviews.

### Columns

* **asin** - ID of the product, such as `B000FA64PK`.
* **helpful** - Helpfulness rating of the review, such as `2/3`.
* **overall** - Rating given to the product.
* **reviewText** - Full text of the review.
* **reviewTime** - Date and time of the review.
* **reviewerID** - ID of the reviewer, such as `A3SPTOKDG7WBLN`.
* **reviewerName** - Name of the reviewer.
* **summary** - Short summary or title of the review.
* **unixReviewTime** - Unix timestamp of the review.

### Acknowledgements

This dataset is taken from the **Amazon Product Data** collected by Julian McAuley at the University of California, San Diego (UCSD).

### Inspiration

* Perform **sentiment analysis** on book reviews.
* Understand what factors influence the **helpfulness** of a review.
* Identify **fake reviews and outliers**.
* Find the **best-rated products**.
* Explore **product similarity** based on reviews.
* Perform other interesting analyses using the review data.


#### Best Practices

1. **Preprocessing and Cleaning**
   - Clean and prepare the review text for analysis.

2. **Train-Test Split**
   - Split the dataset into training and testing sets.

3. **BOW, TF-IDF and Word2Vec**
   - Convert text into numerical features using Bag of Words (BOW), TF-IDF and Word2Vec.

4. **Train ML Algorithms**
   - Train machine learning algorithms using the extracted text features.


In [1]:
import pandas as pd
data = pd.read_csv('../data/kindle_review/all_kindle_review.csv')


In [2]:
data.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [3]:
df = data[['reviewText','rating']]
df.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",3
1,Great short read. I didn't want to put it dow...,5
2,I'll start by saying this is the first of four...,3
3,Aggie is Angela Lansbury who carries pocketboo...,3
4,I did not expect this type of book to be in li...,4


In [4]:
df.shape

(12000, 2)

In [5]:
# Check missing values 
df.isnull().sum()

reviewText    0
rating        0
dtype: int64

In [6]:
# Unique ratings
df['rating'].unique()

array([3, 5, 4, 2, 1])

In [7]:
df['rating'].value_counts()

rating
5    3000
4    3000
3    2000
2    2000
1    2000
Name: count, dtype: int64

In [8]:
# Convert ratings into sentiment labels:
#                   1 = positive review
#                   0 = negative review

df['sentiment'] = df['rating'].apply(lambda rating : 0 if rating<3 else 1)


In [9]:
df['sentiment'].value_counts()

sentiment
1    8000
0    4000
Name: count, dtype: int64

In [10]:
df.head()

,reviewText,rating,sentiment
0,"Jace Rankin may be short, but he's nothing to ...",3,1
1,Great short read. I didn't want to put it dow...,5,1
2,I'll start by saying this is the first of four...,3,1
3,Aggie is Angela Lansbury who carries pocketboo...,3,1
4,I did not expect this type of book to be in li...,4,1


In [11]:
df['sentiment'].unique()

array([1, 0])

#### Preprocessing steps

In [12]:
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\rmahf\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [13]:
! pip install bs4

In [14]:
from bs4 import BeautifulSoup
from nltk.stem import WordNetLemmatizer

##### Text Preprocessing  
- Convert text to lowercase  
- Remove URLs  
- Remove HTML tags  
- Remove special characters  
- Split text into words  
- Remove stopwords  
- Apply lemmatization  
- Join words back into sentences

In [15]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

for i in range(len(df)):
    review = df['reviewText'][i]  # Get the current review
    review = review.lower()  # Convert text to lowercase
    review = re.sub(r'(http|https|ftp|ssh)://([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-])?', '', review)  # Remove URLs
    review = BeautifulSoup(review, 'html.parser').get_text()  # Remove HTML tags  
    review = re.sub('[^a-zA-Z0-9]', ' ', review)  # Remove special characters
    review = review.split()  # Split text into words
    review = [word for word in review if word not in stop_words]  # Remove stopwords
    review = [lemmatizer.lemmatize(word) for word in review]  # Apply lemmatization
    review = ' '.join(review)  # Join words back into a sentence

    df.loc[i, 'reviewText'] = review  # Store the processed review back in DataFrame

In [16]:
df.head()

,reviewText,rating,sentiment
0,jace rankin may short nothing mess man hauled ...,3,1
1,great short read want put read one sitting sex...,5,1
2,start saying first four book expecting conclud...,3,1
3,aggie angela lansbury carry pocketbook instead...,3,1
4,expect type book library pleased find price right,4,1


#### Train-Test Split

In [17]:
from sklearn.model_selection import train_test_split
X_train , X_test, y_train, y_test = train_test_split(df['reviewText'], df['sentiment'], test_size=0.2)

##### Create Bag Of Words

In [18]:
from sklearn.feature_extraction.text import CountVectorizer
bow = CountVectorizer()
X_train_bow = bow.fit_transform(X_train).toarray()
X_test_bow = bow.transform(X_test).toarray()



##### Create TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf_idf = TfidfVectorizer()
X_train_tf_idf = tf_idf.fit_transform(X_train).toarray()
X_test_tf_idf = tf_idf.transform(X_test).toarray()

In [20]:
X_train_bow

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(9600, 24396))

#### Create Word2vec(CBOW, SKIPGRAM) and AvgWord2vec

In [28]:
# Tokenize the data
X_train_tokens = [sentence.split() for sentence in X_train]
X_test_tokens = [sentence.split() for sentence in X_test] 


# Train CBOW model
from gensim.models import Word2Vec
cbow_model =Word2Vec(
    sentences = X_train_tokens,
    vector_size = 100,  # Each word will have 100 numbers
    window = 5,         # Context window size
    min_count = 3,      # Ignore words appearing less than 3 time
    workers = 12,       # uses all 12 cores
    sg = 0,             # 0 = CBOW, 1 = Skip-Gram
    epochs = 15         # Train the model 15 times=10 
)


# Train Skip-Gram model
from gensim.models import Word2Vec

skipgram_model = Word2Vec(
    sentences = X_train_tokens,   # Training sentences
    vector_size = 100,            # Each word will have 100 numbers
    window = 5,                   # Context window size
    min_count =3 ,                # Ignore words appearing less than 3 times
    workers = 12,                 # Uses all 12 cores
    sg = 1,                       # 0 = CBOW, 1 = Skip-Gram
    epochs = 15                   # Train the model 15 times
)


# AVERAGE WORD2VEC (document-level vector)
import numpy as np

def avg_word2vec(document, model):
    known_words = [word for word in document if word in model.wv.index_to_key]

    if not known_words:
        return np.zeros(model.vector_size)
    
    word_vectors = [model.wv[word] for word in known_words]
    avg_vector = np.mean(word_vectors, axis=0)
    return avg_vector

# Generate CBOW-based document vectors (train and test)
X_train_cbow = np.array([avg_word2vec(doc, cbow_model) for doc in X_train_tokens])
X_test_cbow = np.array([avg_word2vec(doc, cbow_model) for doc in X_test_tokens])

# Generate Skip-gram-based document vectors (train and test)
X_train_skipgram = np.array([avg_word2vec(doc, skipgram_model) for doc in X_train_tokens])
X_test_skipgram = np.array([avg_word2vec(doc, skipgram_model) for doc in X_test_tokens])

In [32]:
X_test_cbow

array([[-0.2541924 ,  0.5177643 , -0.33357918, ..., -0.5402482 ,
         0.40108016, -0.3588201 ],
       [-0.1211077 ,  0.6197852 , -0.22250867, ...,  0.07675755,
        -0.21242355, -0.29328564],
       [-0.5140795 ,  0.89576817, -0.08955157, ..., -0.11966731,
        -0.25761226, -0.03546245],
       ...,
       [-0.24907199,  0.4383821 , -0.39572233, ...,  0.10173637,
         0.03170504, -0.34650156],
       [-0.42331374,  0.22718707, -0.19134447, ..., -0.42856848,
        -0.00790251, -0.78546214],
       [-0.15895891,  0.41054118, -0.24235371, ..., -0.12433024,
        -0.00838214, -0.11265652]], shape=(2400, 100), dtype=float32)

In [31]:
X_test_cbow.shape

(2400, 100)

In [30]:
X_train_skipgram.shape

(9600, 100)

##### TRAIN NAIVE BAYES MODELS ON EACH FEATURE SET

In [39]:
from sklearn.naive_bayes import MultinomialNB, GaussianNB

# BOW - MultinomialNB 
naive_bayes_bow_model = MultinomialNB()
naive_bayes_bow_model.fit(X_train_bow, y_train)

# TF-IDF - MultinomialNB 
naive_bayes_tf_idf_model = MultinomialNB()
naive_bayes_tf_idf_model.fit(X_train_tf_idf, y_train)

# Word2Vec CBOW - GaussianNB 
naive_bayes_cbow_model = GaussianNB()
naive_bayes_cbow_model.fit(X_train_cbow, y_train)

# Word2Vec Skip-gram 
naive_bayes_skipgram_model = GaussianNB()
naive_bayes_skipgram_model.fit(X_train_skipgram, y_train)

,"priors priors: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
,"var_smoothing var_smoothing: float, default=1e-9Portion of the largest variance of all features that is added tovariances for calculation stability... versionadded:: 0.20",1e-09
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)number of training samples observed in each class.","ndarray[float32](2,)","[3164.,6436.]"
"class_prior_ class_prior_: ndarray of shape (n_classes,)probability of each class.","ndarray[float32](2,)","[0.33,0.67]"
"classes_ classes_: ndarray of shape (n_classes,)class labels known to the classifier.","ndarray[int64](2,)","[0,1]"
epsilon_ epsilon_: floatabsolute additive value to variances.,float32,np.float32(1.0082254e-11)
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,100
"theta_ theta_: ndarray of shape (n_classes, n_features)mean of each feature per class.","ndarray[float32](2, 100)","[[-0.2 , 0.15,-0.01,...,-0.12, 0.18,-0.14], [-0.18, 0.17, 0.02,...,-0.11, 0.22,-0.14]]"
"var_ var_: ndarray of shape (n_classes, n_features)Variance of each feature per class... versionadded:: 1.0","ndarray[float32](2, 100)","[[0.01,0. ,0. ,...,0.01,0. ,0.01], [0.01,0.01,0. ,...,0.01,0.01,0.01]]"


##### PREDICT AND EVALUATE ACCURACY FOR EACH MODEL

In [41]:
from sklearn.metrics import accuracy_score

y_preds_bow = naive_bayes_bow_model.predict(X_test_bow)
y_preds_tfidf = naive_bayes_tf_idf_model.predict(X_test_tf_idf)
y_preds_cbow = naive_bayes_cbow_model.predict(X_test_cbow)
y_preds_skipgram = naive_bayes_skipgram_model.predict(X_test_skipgram)

print("BOW (MultinomialNB) Accuracy:", accuracy_score(y_test, y_preds_bow))
print("TF-IDF (MultinomialNB) Accuracy:", accuracy_score(y_test, y_preds_tfidf))
print("CBOW (GaussianNB) Accuracy:", accuracy_score(y_test, y_preds_cbow))
print("Skip-gram (GaussianNB) Accuracy:", accuracy_score(y_test, y_preds_skipgram))

BOW (MultinomialNB) Accuracy: 0.8366666666666667
TF-IDF (MultinomialNB) Accuracy: 0.8366666666666667
CBOW (GaussianNB) Accuracy: 0.7445833333333334
Skip-gram (GaussianNB) Accuracy: 0.76375


In [ ]:
from sklearn.metrics import confusion_matrix


cm_bow = confusion_matrix(y_test, y_preds_bow)
print("BOW (MultinomialNB) Confusion Matrix:\n", cm_bow)


cm_tfidf = confusion_matrix(y_test, y_preds_tfidf)
print("TF-IDF (MultinomialNB) Confusion Matrix:\n", cm_tfidf)


cm_cbow = confusion_matrix(y_test, y_preds_cbow)
print("CBOW (GaussianNB) Confusion Matrix:\n", cm_cbow)


cm_skipgram = confusion_matrix(y_test, y_preds_skipgram)
print("Skip-gram (GaussianNB) Confusion Matrix:\n", cm_skipgram)

BOW (MultinomialNB) Confusion Matrix:
 [[ 641  195]
 [ 197 1367]]
TF-IDF (MultinomialNB) Confusion Matrix:
 [[ 641  195]
 [ 197 1367]]
CBOW (GaussianNB) Confusion Matrix:
 [[ 669  167]
 [ 446 1118]]
Skip-gram (GaussianNB) Confusion Matrix:
 [[ 699  137]
 [ 430 1134]]
